In [ ]:
# =========================================================
# EfficientNet-B2 + Augmentações + EarlyStopping + Scheduler
# Fine-tuning parcial (último bloco com LR menor)
# =========================================================
!pip -q install torch torchvision scikit-learn

import os, random, warnings, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
warnings.filterwarnings("ignore")

import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, f1_score, classification_report

# ----------------------------
# Reprodutibilidade básica
# ----------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()
print("Device:", device, "| AMP:", use_amp)

# ============================
# Dataset
# ============================
DATA_ROOT = Path('/home/gustavom/Documents/Orange/Brain Tumor MRI Dataset/Training')
assert DATA_ROOT.exists(), f"Pasta {DATA_ROOT} não encontrada."

use_predefined_split = (DATA_ROOT/"train").exists() and (DATA_ROOT/"validation").exists()

Implementando um Data augmentation mais robusto

In [ ]:
# ============================
# Transforms (robustos)
# - EfficientNet-B2 trabalha em 288x288; usamos resize/crop nessa faixa
# - Normalização padrão ImageNet
# ============================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(288, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.05, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3))
])

val_tfms = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(288),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

if use_predefined_split:
    train_ds = datasets.ImageFolder(DATA_ROOT/"train", transform=train_tfms)
    val_ds   = datasets.ImageFolder(DATA_ROOT/"validation", transform=val_tfms)
    class_names = train_ds.classes
else:
    base_no_tfms = datasets.ImageFolder(DATA_ROOT)   # sem transform para ler rótulos
    class_names = base_no_tfms.classes
    targets = [y for _, y in base_no_tfms.samples]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(sss.split(np.zeros(len(targets)), targets))

    train_all = datasets.ImageFolder(DATA_ROOT, transform=train_tfms)
    val_all   = datasets.ImageFolder(DATA_ROOT, transform=val_tfms)
    train_ds = Subset(train_all, train_idx)
    val_ds   = Subset(val_all,   val_idx)

print("Classes:", class_names)

batch_size = 16
num_workers = 2 if torch.cuda.is_available() else 0
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())

Descongelando camadas

In [ ]:
# ============================
# Modelo: EfficientNet-B2 pré-treinada
# ============================
weights = EfficientNet_B2_Weights.IMAGENET1K_V1
model = efficientnet_b2(weights=weights)

# Congela tudo inicialmente
for p in model.features.parameters():
    p.requires_grad = False

# Descongela APENAS o último bloco (fine-tuning parcial)
for p in model.features[-1].parameters():
    p.requires_grad = True

# Cabeça para 4 classes
num_classes = 4
in_feats = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_feats, num_classes)

model = model.to(device)

Configura Scheduler e Early Stopping
Scheduler reduz o LR quando o loss para de cair (Patience = 2)
Early Stopping para o treinamento quando o loss para de cair (Patience = 4)

In [ ]:
# ============================
# Otimizador com param groups
# - LR menor para último bloco conv
# - LR maior para a cabeça (classifier)
# ============================
last_block_params = [p for p in model.features[-1].parameters() if p.requires_grad]
head_params       = [p for p in model.classifier.parameters() if p.requires_grad]

optimizer = torch.optim.Adam([
    {"params": last_block_params, "lr": 5e-4, "weight_decay": 1e-5},
    {"params": head_params,       "lr": 1e-3, "weight_decay": 1e-5},
])

criterion = nn.CrossEntropyLoss()

# Scheduler: reduz LR quando val_loss "empaca"
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)

scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# ============================
# Early Stopping helper
# ============================
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0, path="best_efficientnetb2.pth"):
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.best = float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best - self.min_delta:
            self.best = val_loss
            self.counter = 0
            self.best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            torch.save(self.best_state, self.path)
            return False  # não para
        else:
            self.counter += 1
            return self.counter > self.patience

early_stopper = EarlyStopping(patience=4, min_delta=1e-4, path="best_efficientnetb2.pth")

In [ ]:
# ============================
# Loops utilitários
# ============================
def run_epoch(model, loader, train=True):
    model.train(mode=train)
    total_loss, total_correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []

    for X, y in loader:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(X)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                logits = model(X)
                loss = criterion(logits, y)

        preds = logits.argmax(1)
        total_loss += loss.item() * y.size(0)
        total_correct += (preds == y).sum().item()
        total += y.size(0)
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())

    avg_loss = total_loss / total
    acc = total_correct / total
    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    f1 = f1_score(y_true, y_pred, average='macro')
    return avg_loss, acc, f1, y_true, y_pred

def plot_curve(hist, k1, k2, title, ylab):
    plt.figure(figsize=(6,4))
    plt.plot(hist[k1], label=k1)
    plt.plot(hist[k2], label=k2)
    plt.xlabel("epoch"); plt.ylabel(ylab); plt.title(title); plt.legend(); plt.show()

def plot_cm(cm, classes, normalize=False, title="Confusion Matrix"):
    if normalize:
        cm = cm.astype(float)/cm.sum(axis=1, keepdims=True)
    import itertools
    plt.figure(figsize=(6,4))
    plt.imshow(cm, interpolation='nearest'); plt.title(title); plt.colorbar()
    ticks = np.arange(len(classes)); plt.xticks(ticks, classes, rotation=45); plt.yticks(ticks, classes)
    fmt = ".2f" if normalize else "d"; thresh = cm.max()/2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i,j], fmt), ha="center",
                 color="white" if cm[i,j] > thresh else "black")
    plt.ylabel("True"); plt.xlabel("Pred"); plt.tight_layout(); plt.show()

In [ ]:
# ============================
# Treinamento + validação
# ============================
EPOCHS = 30  # com early stopping, podemos colocar algo mais alto
history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[], "val_f1":[]}

stop = False
for e in range(1, EPOCHS+1):
    tr_loss, tr_acc, _, _, _ = run_epoch(model, train_loader, train=True)
    va_loss, va_acc, va_f1, y_true, y_pred = run_epoch(model, val_loader, train=False)

    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(va_acc); history["val_f1"].append(va_f1)

    # Scheduler por métrica de validação
    scheduler.step(va_loss)

    print(f"[{e}/{EPOCHS}] "
          f"train_loss={tr_loss:.4f} acc={tr_acc:.4f} | "
          f"val_loss={va_loss:.4f} acc={va_acc:.4f} f1={va_f1:.4f}")

    # Early stopping (salva o melhor estado)
    if early_stopper.step(va_loss, model):
        print(f"Early stopping ativado na época {e}. Melhor val_loss: {early_stopper.best:.4f}")
        stop = True
        break

# Carrega o melhor modelo salvo (pesos)
best_path = "best_efficientnetb2.pth"
if os.path.exists(best_path):
    state = torch.load(best_path, map_location=device)
    model.load_state_dict(state)
    print("Carregado melhor modelo:", best_path)

In [ ]:
# ============================
# Curvas e métricas finais
# ============================
plot_curve(history, "train_loss", "val_loss", "Learning curve (loss)", "loss")
plot_curve(history, "train_acc",  "val_acc",  "Accuracy curve",       "accuracy")

val_loss, val_acc, val_f1, y_true, y_pred = run_epoch(model, val_loader, train=False)
print("\nValidation metrics (best model):")
print(f"Val Loss: {val_loss:.4f} | Accuracy: {val_acc:.4f} | F1 (macro): {val_f1:.4f}")
print("\nClassification report:\n", classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plot_cm(cm, class_names, normalize=False, title="Matriz de Confusão (Bruta)")
plot_cm(cm, class_names, normalize=True,  title="Matriz de Confusão (Normalizada)")